# SmartEvent: Machine Learning Attendance Prediction Model
**Institution:** Strathmore University — School of Computing and Engineering Sciences  
**System:** SmartEvent Attendance Prediction Engine  

---  
### Overview
This Google Colab notebook processes the `Attendance_Prediction.csv` dataset (20,000 student records):
1. **Google Drive Integration**: Mounts Google Drive and reads `Attendance_Prediction.csv`.
2. **Data Preprocessing & Feature Engineering**: Maps student attributes (`course`, `class_type`, `weather`, `travel_time`, `study_hours`) into attendance prediction features.
3. **Model Training & Evaluation**: Trains Random Forest models with evaluation metrics (MAE, R² score).
4. **Export to Google Drive**: Saves `attendance_model.joblib` directly to `/content/drive/MyDrive/SmartEvent_ML/models/`.

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE_DIR = '/content/drive/MyDrive/SmartEvent_ML'
DATASET_DIR = os.path.join(DRIVE_BASE_DIR, 'datasets')
MODEL_DIR = os.path.join(DRIVE_BASE_DIR, 'models')

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Google Drive Mounted Successfully!")

## Step 2: Load Attendance_Prediction.csv Dataset

In [ ]:
import pandas as pd
import numpy as np

dataset_path = os.path.join(DATASET_DIR, 'Attendance_Prediction.csv')
if not os.path.exists(dataset_path):
    # Fallback to local notebook directory
    dataset_path = 'Attendance_Prediction.csv'

print(f"Reading dataset from: {dataset_path}")
df = pd.read_csv(dataset_path)
print(f"Loaded {len(df)} student attendance records!")
df.head()

## Step 3: Feature Engineering & Model Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Feature Mapping
df['eventType'] = df['course'].astype(str).str.upper()
df['timeSlot'] = df['class_type'].apply(lambda x: 'Morning' if str(x).lower() == 'offline' else 'Afternoon')
df['dayOfWeek'] = df['travel_time_minutes'].apply(lambda x: int(x) % 7)
df['venueCapacity'] = df['study_hours'].apply(lambda x: int(50 + (x * 30)))
df['registrationCount'] = df['venueCapacity'].apply(lambda c: int(c * 0.85))
df['pastAttendanceRate'] = df['attendance'].apply(lambda a: 0.85 if a == 1 else 0.55)
df['smsPriority'] = df['weather'].apply(lambda w: 3 if str(w).lower() in ['rainy', 'cold'] else 2)
df['attendanceCount'] = df.apply(lambda r: int(r['registrationCount'] * (0.85 if r['attendance'] == 1 else 0.50)), axis=1)

X = df[['eventType', 'dayOfWeek', 'timeSlot', 'venueCapacity', 'registrationCount', 'pastAttendanceRate', 'smsPriority']]
y = df['attendanceCount']

categorical_features = ['eventType', 'timeSlot']
numerical_features = ['dayOfWeek', 'venueCapacity', 'registrationCount', 'pastAttendanceRate', 'smsPriority']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ]
)

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)

predictions = model_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"MAE: {mae:.2f} attendees")
print(f"R² Score: {r2:.4f}")

## Step 4: Save Model to Google Drive

In [ ]:
import joblib

output_model_path = os.path.join(MODEL_DIR, 'attendance_model.joblib')
joblib.dump(model_pipeline, output_model_path)
print(f"Saved model to Google Drive: {output_model_path}")